# MCXN236 Secure Boot - out-of-box workflow with recovery

This notebook walks through the complete secure boot workflow on the **FRDM-MCXN236** board using SPSDK and shows how to get out of the situations that typically look like a "dead" device:

- [generate the Root of Trust (RoT) key](#generate-the-root-of-trust-key),
- [create the certificate block and the signed Master Boot Image (MBI)](#certificate-block-and-signed-master-boot-image),
- [prepare the CMPA that **enables secure boot**](#cmpa-with-secure-boot-enabled) (`SECURE_BOOT_CFG.SEC_BOOT_EN`) and check it against a pre-write checklist,
- [write the CMPA and the signed image and boot it](#write-the-cmpa-and-the-signed-image),
- [load an image from the debugger / IDE](#loading-an-image-from-the-debugger-ide) (and see why an unsigned image is rejected),
- [recover the device](#recovery) - non-destructively first, by Mass Erase as the last option.

The notebook also describes the **Mass Erase side effects** on MCXN236.

<div class="alert alert-block alert-danger">
<b>WARNING:</b> This example really enables secure boot in the CMPA of the device. After the next reset the boot ROM executes <b>only images signed by the key generated in this notebook</b>. Read the whole notebook, in particular the <a href="#4.2-Pre-write-checklist">pre-write checklist</a> and the <a href="#7.-Recovery">recovery</a> section, <b>before</b> running the cells that write to the device. Never use the keys from this example in production.
</div>

## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- FRDM-MCXN236 board connected to the PC:
  - USB cable to the MCU-Link debug probe connector - provides the debug probe (for `nxpdebugmbox`) and a virtual COM port (UART ISP),
  - optionally a second USB cable to the MCU USB connector (USB ISP).
- The device must be in the **Develop** life cycle (default state of the FRDM board, `LC_STATE = 0x3`). Do not advance the life cycle while following this example.
- The secure boot related OTP fuses must not be programmed (default state of the FRDM board). The ROM combines them with the CMPA and the OTP value always takes precedence or makes the setting stricter:
  - `SEC_BOOT_EN` - the stricter of the OTP and CMPA value is used,
  - `ROTKH` - if the OTP `ROTKH` is programmed, the CMPA `ROTKH` is ignored,
  - `ENF_CNSA` - when set (in OTP or CMPA), the ROM requires `secp384r1` root keys; the `secp256r1` key used in this example works only with `ENF_CNSA` = 0.

In [1]:
# Initialization cell
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
FAMILY = "mcxn236"
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"  # input configurations and application images
KEYS = WORKSPACE + "keys/"  # keys generated by this notebook
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

## 2. Generate the Root of Trust key

Secure boot is anchored by up to four Root of Trust keys (RoTK). The hash of the root key table (RKTH) is written into CMPA (`ROTKH`); from that moment only images signed by one of these keys are accepted. In this example we use **one** `secp256r1` key generated by the *nxpcrypto* application (see [How-to-get-keys-using-nxpcrypto](../../crypto/keys.ipynb)).

<div class="alert alert-block alert-warning">
<b>Keep the private key!</b> If the private key is lost after the CMPA with secure boot is written, no new image can be signed for the device and the only way back is the <a href="#7.-Recovery">recovery</a> procedure.
</div>

In [3]:
import os

ROTK0_PRV = KEYS + "rotk0_p256.pem"
%! nxpcrypto $VERBOSITY key generate -k secp256r1 -o $ROTK0_PRV --force
assert os.path.exists(ROTK0_PRV)

nxpcrypto  key generate -k secp256r1 -o workspace/keys/rotk0_p256.pem --force 
The key pair has been created: workspace/keys/rotk0_p256.pub, workspace/keys/rotk0_p256.pem


## 3. Certificate block and signed Master Boot Image

The certificate block holds the root key table. It is referenced by the MBI configuration and by `pfr export` (to fill `ROTKH` in CMPA), so the RKTH in the image and in the CMPA always match. Configuration: [cert_block_mcxn236.yaml](./inputs/cert_block_mcxn236.yaml).

The signed MBI is built from a plain *hello world* application (green LED blinking). The difference between the `signed`/`xip` template and the user configuration [mbi_signed.yaml](./inputs/mbi_signed.yaml) is shown below.

In [4]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/mbi_signed.diffc").html

nxpimage mbi get-templates -f mcxn236 -o workspace --force 
Creating workspace/mcxn236_xip_plain.yaml template file.
Creating workspace/mcxn236_xip_crc.yaml template file.
Creating workspace/mcxn236_xip_signed.yaml template file.
Creating workspace/mcxn236_load_to_ram_plain.yaml template file.
Creating workspace/mcxn236_load_to_ram_crc.yaml template file.
Creating workspace/mcxn236_load_to_ram_signed.yaml template file.


In [5]:
CERT_BLOCK_CONFIG = INPUTS + "cert_block_mcxn236.yaml"
MBI_SIGNED_CONFIG = INPUTS + "mbi_signed.yaml"
MBI_PLAIN_CONFIG = INPUTS + "mbi_plain.yaml"
SIGNED_IMAGE = WORKSPACE + "mcxn236_signed_green.bin"
PLAIN_IMAGE = WORKSPACE + "mcxn236_plain_red.bin"

%! nxpimage $VERBOSITY cert-block export -c $CERT_BLOCK_CONFIG
assert os.path.exists(WORKSPACE + "cert_block.bin")
%! nxpimage $VERBOSITY mbi export -c $MBI_SIGNED_CONFIG
assert os.path.exists(SIGNED_IMAGE)
# Plain (unsigned) image - used later only to demonstrate the "dead device" symptom
%! nxpimage $VERBOSITY mbi export -c $MBI_PLAIN_CONFIG
assert os.path.exists(PLAIN_IMAGE)

nxpimage  cert-block export -c inputs/cert_block_mcxn236.yaml 
RKTH: 6897cdf4d7acffc09ef7d7ab3daca5d15997280833ae2ea5e6c056bd2ed3628d
Success. (Certificate Block: workspace/cert_block.bin created.)


nxpimage  mbi export -c inputs/mbi_signed.yaml 
RKTH: 6897cdf4d7acffc09ef7d7ab3daca5d15997280833ae2ea5e6c056bd2ed3628d
Success. (Master Boot Image: workspace/mcxn236_signed_green.bin created.)
PFR CMPA configuration with the MBI RKTH has been stored in: workspace/mbi_pfr_cmpa.yaml


nxpimage  mbi export -c inputs/mbi_plain.yaml 
Success. (Master Boot Image: workspace/mcxn236_plain_red.bin created.)


## 4. CMPA with secure boot enabled

### 4.1 CMPA configuration

The *pfr* application generates the CMPA binary. Only two things are changed against the defaults (see [cmpa_mcxn236.yaml](./inputs/cmpa_mcxn236.yaml)):

- `SECURE_BOOT_CFG.SEC_BOOT_EN = SIGNED_IMAGES` - the boot ROM accepts only signed images,
- `RoTK_USAGE` - only RoTK0 is used, slots 1..3 are marked as not used.

`ROTKH` is filled automatically from the certificate block (`-e` option).

The ROM uses the CMPA only if its header marker (`BOOT_CFG.CMPA_HEADER = 0x5963`) is set; otherwise the whole page is ignored. *pfr* sets the marker by default, so keep it when editing the CMPA manually.

| `SEC_BOOT_EN` | Accepted images |
|---------------|-----------------|
| `ALL_IMAGES` (0, default) | plain, CRC, signed |
| `CRC32_IMAGES` (1) | CRC32 or signed |
| `SIGNED_IMAGES` (2) | signed only |
| `ECDSA_SIGNED` (3) | ECDSA signed only |

In [6]:
# Get difference of template and user YAML configuration of CMPA
YamlDiffWidget("inputs/cmpa_mcxn236.diffc").html

pfr get-template -t cmpa -f mcxn236 -o workspace/cmpa_mcxn236_template.yaml --force 
The PFR cmpa template for mcxn236, Revision: latest has been saved into workspace/cmpa_mcxn236_template.yaml YAML file


In [7]:
CMPA_CONFIG = INPUTS + "cmpa_mcxn236.yaml"
CMPA_BIN = WORKSPACE + "cmpa_mcxn236.bin"
CMPA_PARSED = WORKSPACE + "cmpa_mcxn236_parsed.yaml"
%! pfr $VERBOSITY export -c $CMPA_CONFIG -e $CERT_BLOCK_CONFIG -o $CMPA_BIN
assert os.path.exists(CMPA_BIN)
# Show what differs from the defaults in the final binary - this is exactly what will be written
%! pfr $VERBOSITY parse -f $FAMILY -t cmpa -b $CMPA_BIN -d -o $CMPA_PARSED
assert os.path.exists(CMPA_PARSED)

pfr  export -c inputs/cmpa_mcxn236.yaml -e inputs/cert_block_mcxn236.yaml -o workspace/cmpa_mcxn236.bin 
Success. (CMPA binary has been generated and stored into workspace/cmpa_mcxn236.bin)


pfr  parse -f mcxn236 -t cmpa -b workspace/cmpa_mcxn236.bin -d -o workspace/cmpa_mcxn236_parsed.yaml 
Success. (PFR/IFR: workspace/cmpa_mcxn236.bin has been parsed.


### 4.2 Pre-write checklist

<div class="alert alert-block alert-danger">
<b>WARNING - read before writing the CMPA.</b> After <code>pfr write -t cmpa</code> and a reset:
<ul>
<li>the boot ROM refuses <b>every unsigned image</b> - an application downloaded by MCUXpresso IDE / VS Code / IAR from a plain <code>.axf</code>/<code>.elf</code> does not start and the debug session looks "dead",</li>
<li>the board falls back to ISP mode (if ISP is not disabled) instead of running the application,</li>
<li>recovery is possible only with a correctly signed image, by erasing the CMPA, or by Mass Erase (see <a href="#7.-Recovery">Recovery</a>).</li>
</ul>
</div>

Before writing, make sure that:

- [ ] the private key `workspace/keys/rotk0_p256.pem` is stored safely,
- [ ] `ROTKH` in the CMPA binary equals the RKTH of the certificate block (checked by the cell below),
- [ ] the signed image is built and is signed by the same key (checked by the cell below),
- [ ] the debug interface is **not** locked (`CC_SOCU_PIN` / `CC_SOCU_DFLT` kept on defaults) and the Mass Erase debug mailbox command is not disabled - otherwise the debugger-based recovery is not possible,
- [ ] ISP entry is **not** disabled (`FLASH_CFG.ISP_PIN_ENTRY`, `ISP_DM_ENTRY`, `ISP_API_ENTRY` and `ISP_FT_ENTRY` kept on defaults) - the device can still be reached by *blhost* / *pfr*. `ISP_FT_ENTRY` is the most important one: if ISP entry after an image authentication failure is disabled, the ROM stops in an endless loop instead of entering ISP mode and only the debug mailbox (ISP mode command or Mass Erase) can recover the device,
- [ ] the life cycle is **not** advanced (no `program-lifecycle`, no life cycle `programFuses` command) - while the device is in Develop, the CMPA can be rewritten and Mass Erase restores it.

The cell below performs the automatic part of the checklist and stops the notebook if something does not fit.

In [8]:
from spsdk.image.cert_block.cert_block_v21 import CertBlockV21
from spsdk.pfr.pfr import CMPA
from spsdk.utils.family import FamilyRevision
from spsdk.utils.misc import load_binary

cmpa = CMPA.parse(load_binary(CMPA_BIN), FamilyRevision(FAMILY))
cert_block = CertBlockV21.parse(load_binary(WORKSPACE + "cert_block.bin"))

sec_boot = cmpa.registers.find_reg("SECURE_BOOT_CFG").find_bitfield("SEC_BOOT_EN")
rotkh = cmpa.registers.find_reg("ROTKH", include_group_regs=True).get_hex_value()
rkth = cert_block.rkth.hex()

print(f"SEC_BOOT_EN : {sec_boot.get_enum_value()} ({sec_boot.get_value()})")
print(f"CMPA ROTKH  : {rotkh.lower()}")
print(f"Cert RKTH   : {rkth}")
assert rotkh.lower() == rkth, "ROTKH in CMPA does not match the certificate block!"
assert load_binary(SIGNED_IMAGE), "Signed image is missing!"
if sec_boot.get_value() >= 2:
    print("WARNING: secure boot will be ENFORCED - only signed images will boot after reset.")
print("Pre-write checks passed.")

SEC_BOOT_EN : SIGNED_IMAGES (2)
CMPA ROTKH  : 6897cdf4d7acffc09ef7d7ab3daca5d15997280833ae2ea5e6c056bd2ed3628d
Cert RKTH   : 6897cdf4d7acffc09ef7d7ab3daca5d15997280833ae2ea5e6c056bd2ed3628d


Pre-write checks passed.


## 5. Write the CMPA and the signed image

### 5.1 Connect the board in ISP mode

The device has to be in ISP mode to be reachable by *blhost* / *pfr*. There are two ways:

- hold the **ISP** button (pin `P0_6`, ISP mode is selected when the pin is low during reset), press and release **RESET**, then release **ISP**,
- use the debug mailbox over the MCU-Link debug probe: `nxpdebugmbox -i pyocd cmd -f mcxn236 ispmode -m 0`. This works also when a valid application is running and is used by this notebook.

Use *nxpdevscan* to find the UART (`FLEXCOMM4`, pins `P1_8`/`P1_9`, connected to the MCU-Link VCOM) or USB (`0x1FC9:0x0158`) ISP interface.

In [9]:
DEBUG_PROBE = "-i pyocd"  # debug probe interface used by nxpdebugmbox

# Force ISP mode over the debug mailbox (alternatively use the ISP + RESET buttons)
%! nxpdebugmbox $VERBOSITY $DEBUG_PROBE cmd -f $FAMILY ispmode -m 0
# check if the device is connected and detected by the PC
%! nxpdevscan

nxpdebugmbox  -i pyocd cmd -f mcxn236 ispmode -m 0 
  #   Interface   Id              Description                                                        Architecture   Protocols  
-------------------------------------------------------------------------------------------------------------------------------
  0   PyOCD       UGOCKDKFN42SB   NXP Semiconductors MCU-LINK FRDM-MCXN236 (r0E7) CMSIS-DAP V3.130   arm-cortex     swd, jtag  
Entering into ISP mode succeeded


nxpdevscan 
Scanning USB devices...
╔═════════════════════════════════════════════════════════════╗
║            Connected NXP USB Devices  (2 devices)           ║
╠═════════════════════════════════════════════════════════════╣
║  VID       : 0x1fc9                                         ║
║  PID       : 0x0143                                         ║
║  Product   : LPCSIO                                         ║
║  Name      : N/A                                            ║
║  Path      : HID\VID_1FC9&PID_0143&MI_04\7&1775498E&0&0000  ║
║  Path Hash : 15f7febd                                       ║
║  Serial    : UGOCKDKFN42SB                                  ║
╟─────────────────────────────────────────────────────────────╢
║  VID       : 0x1fc9                                         ║
║  PID       : 0x0143                                         ║
║  Product   : MCU-LINK NXP TRACE                             ║
║  Name      : N/A                                            ║
║  P

In [10]:
# Select the interface to the target based on the result of nxpdevscan.
# USB ISP: "-u 0x1fc9,0x0158"; UART ISP via MCU-Link VCOM: "-p comXX"
INTERFACE = "-p com22"  # change to your device

%! blhost $INTERFACE get-property current-version
# Property 17 (security state) returns two words: the secure state and the life cycle value.
# The second word is the life cycle state - must be 0x3 (Develop) for this example
%! blhost $INTERFACE get-property 17

blhost -p com22 get-property current-version 
Response status = 0 (0x0) Success.
Response word 1 = 1258488320 (0x4b030200)
Current Version = K3.2.0


blhost -p com22 get-property 17 
Response status = 0 (0x0) Success.
Response word 1 = 1520786085 (0x5aa55aa5)
Response word 2 = 3 (0x3)
Security State = UNSECURE


### 5.2 Write the signed image first, then the CMPA

The signed image is written **before** the CMPA - if anything goes wrong in between, the device still has a valid image for the moment secure boot becomes active.

In [11]:
%! blhost $INTERFACE flash-erase-all
%! blhost $INTERFACE write-memory 0x0 $SIGNED_IMAGE

blhost -p com22 flash-erase-all 
Response status = 0 (0x0) Success.


blhost -p com22 write-memory 0x0 workspace/mcxn236_signed_green.bin 
Response status = 0 (0x0) Success.
Response word 1 = 15112 (0x3b08)


<div class="alert alert-block alert-danger">
<b>POINT OF NO SILENT RETURN:</b> the next cell writes the CMPA with <code>SEC_BOOT_EN = SIGNED_IMAGES</code>. From the next reset only images signed by <code>rotk0_p256.pem</code> are executed. The <code>pfr write</code> command does <b>not</b> ask for confirmation.
</div>

In [12]:
%! pfr $VERBOSITY write $INTERFACE -t cmpa -f $FAMILY -bin $CMPA_BIN
%! blhost $INTERFACE reset

pfr  write -p com22 -t cmpa -f mcxn236 -bin workspace/cmpa_mcxn236.bin 
CMPA data written to device.


blhost -p com22 reset 
Response status = 0 (0x0) Success.


After the reset the board runs the signed *hello world* application: the **green** LED blinks and `hello world green` is printed on the MCU-Link VCOM.

## 6. Loading an image from the debugger / IDE

With secure boot enabled, an IDE (MCUXpresso IDE, MCUXpresso for VS Code, IAR, Keil) that downloads the **plain** `.axf`/`.elf` into flash creates an image that the boot ROM rejects. The debugger halts in the ROM or the board ends in ISP mode and the session looks "dead". This is the expected behavior, not a broken device.

To debug with secure boot enabled:

- build the application, convert it to a raw `.bin` linked at `0x0`, and sign it with `nxpimage mbi export` (the same configuration as in this notebook, only `inputImageFile` changes),
- program the **signed** binary (IDE flash programmer "program binary" / *blhost* `write-memory`),
- start the debug session in **attach** mode (do not let the IDE download the plain `.axf` again) and load only the symbols from the `.axf`.

### 6.1 Demonstration: unsigned image is rejected

The signed application is running now, so the next cell first forces ISP mode over the debug mailbox. Then it writes the plain (unsigned, red LED) image, as an IDE would do. After the reset the application does **not** start, the red LED does not blink and the ROM falls back to ISP mode - therefore *blhost* still answers.

In [13]:
# The signed application is running - enter ISP mode over the debug mailbox first
%! nxpdebugmbox $VERBOSITY $DEBUG_PROBE cmd -f $FAMILY ispmode -m 0
%! blhost $INTERFACE flash-erase-all
%! blhost $INTERFACE write-memory 0x0 $PLAIN_IMAGE
%! blhost $INTERFACE reset

nxpdebugmbox  -i pyocd cmd -f mcxn236 ispmode -m 0 
  #   Interface   Id              Description                                                        Architecture   Protocols  
-------------------------------------------------------------------------------------------------------------------------------
  0   PyOCD       UGOCKDKFN42SB   NXP Semiconductors MCU-LINK FRDM-MCXN236 (r0E7) CMSIS-DAP V3.130   arm-cortex     swd, jtag  
Entering into ISP mode succeeded


blhost -p com22 flash-erase-all 
Response status = 0 (0x0) Success.


blhost -p com22 write-memory 0x0 workspace/mcxn236_plain_red.bin 
Response status = 0 (0x0) Success.
Response word 1 = 14944 (0x3a60)


blhost -p com22 reset 
Response status = 0 (0x0) Success.


In [14]:
# The ROM rejected the unsigned image and entered ISP mode -> the device still responds
%! blhost $INTERFACE get-property current-version

blhost -p com22 get-property current-version 
Response status = 0 (0x0) Success.
Response word 1 = 1258488320 (0x4b030200)
Current Version = K3.2.0


## 7. Recovery

Use the options in this order - from the least to the most destructive.

### 7.1 Non-destructive: write a correctly signed image

As long as the device is reachable over ISP (it falls back to ISP when the image authentication fails) simply write the signed image again. Secure boot stays enabled. After the reset the **green** LED blinks again.

In [15]:
%! blhost $INTERFACE flash-erase-all
%! blhost $INTERFACE write-memory 0x0 $SIGNED_IMAGE
%! blhost $INTERFACE reset

blhost -p com22 flash-erase-all 
Response status = 0 (0x0) Success.


blhost -p com22 write-memory 0x0 workspace/mcxn236_signed_green.bin 
Response status = 0 (0x0) Success.
Response word 1 = 15112 (0x3b08)


blhost -p com22 reset 
Response status = 0 (0x0) Success.


### 7.2 Non-destructive: return the CMPA to defaults (disable secure boot)

In the **Develop** life cycle the CMPA page is not sealed and can be rewritten. `pfr erase-cmpa` writes the default CMPA (`SEC_BOOT_EN = ALL_IMAGES`) and keeps the flash content. Afterwards unsigned images (e.g. from the IDE) work again - the cell writes the plain image and the **red** LED blinks.

The signed application is running after 7.1, so the cell below enters ISP mode over the debug mailbox first (the ISP + RESET buttons can be used instead).

In [16]:
# Enter ISP mode over the debug mailbox (SWD) - the signed application is running
%! nxpdebugmbox $VERBOSITY $DEBUG_PROBE cmd -f $FAMILY ispmode -m 0

%! pfr $VERBOSITY erase-cmpa $INTERFACE -f $FAMILY
# Secure boot is disabled now - the plain (red LED) image boots again
%! blhost $INTERFACE flash-erase-all
%! blhost $INTERFACE write-memory 0x0 $PLAIN_IMAGE
%! blhost $INTERFACE reset

nxpdebugmbox  -i pyocd cmd -f mcxn236 ispmode -m 0 
  #   Interface   Id              Description                                                        Architecture   Protocols  
-------------------------------------------------------------------------------------------------------------------------------
  0   PyOCD       UGOCKDKFN42SB   NXP Semiconductors MCU-LINK FRDM-MCXN236 (r0E7) CMSIS-DAP V3.130   arm-cortex     swd, jtag  
Entering into ISP mode succeeded


pfr  erase-cmpa -p com22 -f mcxn236 
CMPA page address on mcxn236, Revision: latest is 0x1004000
CMPA page has been erased.


blhost -p com22 flash-erase-all 
Response status = 0 (0x0) Success.


blhost -p com22 write-memory 0x0 workspace/mcxn236_plain_red.bin 
Response status = 0 (0x0) Success.
Response word 1 = 14944 (0x3a60)


blhost -p com22 reset 


Response status = 0 (0x0) Success.


### 7.3 Destructive: Mass Erase over the debug mailbox

If neither ISP nor a signed image helps, Mass Erase over the debug mailbox erases the whole internal flash **and the CMPA** on MCXN236. The device returns to the default boot configuration; the life cycle is not changed.

Prerequisites:

- debug probe connected (MCU-Link on the FRDM board),
- Mass Erase debug mailbox command enabled in CMPA (`CC_SOCU_PIN/CC_SOCU_DFLT.ME_CMD_EN`) - this is required in every life cycle, including Develop,
- life cycle not advanced beyond Develop if you expect the device to be fully open after the erase.

The command in the cell below is commented out on purpose - uncomment it only when you really need the Mass Erase.

In [17]:
# Run only if 7.1 and 7.2 did not help - erases the whole flash and the CMPA
# %! nxpdebugmbox $VERBOSITY $DEBUG_PROBE cmd -f $FAMILY erase

### 7.4 Mass Erase side effects on MCXN236

- In the **Develop** life cycle Mass Erase erases the **flash** and the **CMPA / CFPA** pages and reverts the boot mode to defaults (secure boot disabled).
- In an advanced life cycle Mass Erase erases only the flash - the CMPA (and therefore secure boot) is kept.
- The **life cycle is not affected**. If the life cycle was advanced, the device stays in that life cycle and its restrictions apply after the erase.
- The ROM can store a device-unique CMAC of the image after a successful ECDSA authentication and use it on the following boots. This is requested by the image update field in CFPA and is possible in any OEM life cycle, including Develop. With the default CFPA used in this example no CMAC is stored. If a CMAC was stored, Mass Erase in Develop removes it together with the CFPA.